# 👑 Mixture of Scars (MoS) Titan 3 Standalone: Mistral Large 2 (123B) L44 + SCache O(1)
Zero-leak pure-PyTorch FP16 layer-streaming (`micro_bsz=8`, `60k` tokens/GPU, `< 4.2 GiB` peak VRAM on T4).


In [ ]:
import gc, glob, json, math, os, re, shutil, time
from pathlib import Path
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from safetensors import safe_open
from safetensors.torch import load_file

torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"
W_SVC = 1.0
W_DEBERTA = 0.25
TEMP_TITAN3 = 5.0
W_TITAN3_PLAIN = 0.25
W_TITAN3_TWIN = 0.35
TWIN = dict(thresh=0.89, min_size=2, max_size=18)

LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)

def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]

def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def vote(name, P, w, rows=slice(None)):
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1.0))
    print(f"{name}: weight {w:.3f}, done after {time.time() - T0:.0f}s", flush=True)

def extract_tricompass_style_features(texts):
    feats = []
    for t in texts:
        t = str(t)
        n = max(len(t), 1); words = t.split(); nw = max(len(words), 1)
        sents = [s for s in re.split(r"[.!?]+", t) if s.strip()]; ns = max(len(sents), 1)
        lines = [l for l in t.split("\n") if l.strip()]; nl = max(len(lines), 1)
        feats.append([
            t.count(",")/n, t.count(":")/n, t.count(";")/n, t.count("—")/n, t.count(" — ")/n,
            t.count("-")/n, t.count("--")/n, t.count('"')/n, t.count("'")/n, t.count("`")/n, t.count("```")/nw,
            t.count("(")/n, t.count(")")/n, t.count("[")/n, t.count("]")/n, t.count("{")/n, t.count("}")/n,
            t.count("#")/n, t.count("*")/n, (t.count("\n- ") + t.count("\n* "))/nl,
            sum(c.isdigit() for c in t)/n, sum(c.isupper() for c in t)/n,
            sum(w[0].isupper() for w in words if len(w) > 0)/nw,
            t.count("\n")/nw, t.count("\n\n")/nw,
            sum(len(w) for w in words)/nw, nw/ns, len(set(w.lower() for w in words))/nw
        ])
    return np.asarray(feats, dtype=np.float32)

class TestTimeSCache:
    def __init__(self, slot_dim=32, resonance_scale=0.15):
        self.slot_dim = slot_dim
        self.resonance_scale = resonance_scale
        self.proj = None

    def apply_scache_resonance(self, X, groups, twin_mask=None):
        if self.proj is None:
            rng = np.random.RandomState(42)
            Q, _ = np.linalg.qr(rng.randn(X.shape[1], self.slot_dim))
            self.proj = Q.astype(np.float32)
        Z = X @ self.proj
        X_out = X.copy()
        ugs = np.unique(groups[twin_mask]) if (twin_mask is not None and twin_mask.any()) else []
        for g in ugs:
            m = (groups == g)
            if m.sum() > 1:
                dz = Z[m] - Z[m].mean(axis=0, keepdims=True)
                W_scar = (dz.T @ dz) / float(m.sum())
                X_out[m] += self.resonance_scale * ((dz @ W_scar.T) @ self.proj.T)
        return X_out

"""
System One — Unified Zero-Leak Pure-PyTorch FP16 MoS Layer-Streaming Engine
===========================================================================
Suporta:
1. Qualquer backbone NF4 MoS (Mistral Large 123B, Yi-34B, Qwen3-Embedding-8B, Qwen-72B)
   com detecção automática de prefixo ('model.' vs ''), GQA, RoPE theta e left/right padding.
2. Isolamento estrito de escopo em _step_attn() e _step_mlp(): zero vazamento de tensores
   entre subcamadas (pico de VRAM <= 4.2 GiB por GPU T4 mesmo para N = 10.000 passagens).
3. Extração Multi-Depth simultânea opcional (ex: 8 fatias [4,8,12,16,20,24,30,36]) com
   custo zero de I/O extra + fusão in-place opcional de MoS Residual Scar Experts (dW = B @ A).
4. Janelas deslizantes de cobertura completa (covering windows sem truncamento) com agregação
   ponderada por contagem de tokens válidos.
"""


import gc
import json
import os
import time
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from safetensors import safe_open


def build_covering_windows(
    ids_list: Sequence[Sequence[int]],
    max_len: int = 512,
    stride: int = 384,
) -> Tuple[List[List[int]], np.ndarray, np.ndarray]:
    """
    Decompõe sequências longas em janelas deslizantes de até `max_len` tokens sem
    descartar parágrafos finais. Para sequências <= max_len, gera exatamente 1 janela.
    Retorna:
      win_ids: lista de sequências de tokens por janela
      doc_idx: índice do documento original de cada janela (M,)
      win_weights: peso proporcional ao comprimento útil de cada janela (M,)
    """
    win_ids: List[List[int]] = []
    doc_idx: List[int] = []
    win_weights: List[float] = []

    for i, seq in enumerate(ids_list):
        s = list(seq)
        L = len(s)
        if L <= max_len:
            win_ids.append(s)
            doc_idx.append(i)
            win_weights.append(float(max(L, 1)))
        else:
            starts = list(range(0, L - max_len + 1, stride))
            if starts[-1] + max_len < L:
                starts.append(L - max_len)
            for st in starts:
                chunk = s[st : st + max_len]
                win_ids.append(chunk)
                doc_idx.append(i)
                win_weights.append(float(len(chunk)))

    return win_ids, np.asarray(doc_idx, dtype=np.int64), np.asarray(win_weights, dtype=np.float32)


def aggregate_window_embeddings(
    Z_win: np.ndarray,
    doc_idx: np.ndarray,
    win_weights: np.ndarray,
    n_docs: int,
    unit_rms: bool = True,
) -> np.ndarray:
    """Agrega embeddings de janelas deslizantes de volta para (n_docs, D)."""
    D = Z_win.shape[1]
    out = np.zeros((n_docs, D), dtype=np.float32)
    w_sum = np.zeros((n_docs, 1), dtype=np.float32)
    np.add.at(out, doc_idx, Z_win * win_weights[:, None])
    np.add.at(w_sum[:, 0], doc_idx, win_weights)
    out /= np.maximum(w_sum, 1e-6)
    if unit_rms:
        rms = 1.0 / np.sqrt(np.mean(out * out, axis=-1, keepdims=True) + 1e-6)
        out = out * rms
    return out


def extract_mos_nf4_streaming(
    backbone_dir: str,
    ids_list: Sequence[Sequence[int]],
    pad_id: int,
    devices: Sequence[str],
    padding_side: str = "left",
    tag: str = "MoS",
    micro_bsz: int = 8,
    max_tokens_per_dev: int = 60000,
    capture_layers: Optional[Sequence[int]] = None,
    scar_tensors: Optional[Dict[str, torch.Tensor]] = None,
    scar_scale: float = 1.0,
    log_every: int = 10,
) -> np.ndarray:
    """
    Motor canônico puro-PyTorch FP16 de Layer-Streaming MoS.
    - Se `capture_layers` for None: retorna (N, d_model) da última camada do backbone.
    - Se `capture_layers` for uma lista de índices 1-based (ex: [4,8,12,16,20,24,30,36]):
      retorna (N, len(capture_layers) * d_model) concatenando os pools normalizados!
    """
    devices = list(dict.fromkeys(devices))
    with open(os.path.join(backbone_dir, "config.json"), "r") as f:
        cfg_raw = json.load(f)

    n_layers = int(cfg_raw["num_hidden_layers"])
    d_model = int(cfg_raw["hidden_size"])
    n_heads = int(cfg_raw["num_attention_heads"])
    n_kv = int(cfg_raw["num_key_value_heads"])
    head_dim = int(cfg_raw.get("head_dim") or (d_model // n_heads))
    eps = float(cfg_raw.get("rms_norm_eps", 1e-5))
    rope_params = cfg_raw.get("rope_parameters") or {}
    theta = float(cfg_raw.get("rope_theta") or rope_params.get("rope_theta") or 1000000.0)
    gqa_rep = n_heads // n_kv
    has_qkv_bias = bool(cfg_raw.get("attention_bias", False))

    with open(os.path.join(backbone_dir, "model.safetensors.index.json"), "r") as f:
        idx_map = json.load(f)["weight_map"]

    base_pfx = "model." if "model.embed_tokens.weight" in idx_map else ""
    layers_pfx = f"{base_pfx}layers."
    norm_key = f"{base_pfx}norm.weight"
    open_shards: Dict[str, object] = {}

    target_layers_1b = list(capture_layers) if capture_layers is not None else [n_layers]
    max_target_layer = max(target_layers_1b)
    if max_target_layer > n_layers:
        raise ValueError(f"Requested layer {max_target_layer} exceeds backbone num_hidden_layers={n_layers}")
    capture_map = {l_1b - 1: slot_i for slot_i, l_1b in enumerate(target_layers_1b)}

    def get_tensor_cpu(key: str) -> torch.Tensor:
        shard_name = idx_map[key]
        if shard_name not in open_shards:
            open_shards[shard_name] = safe_open(
                os.path.join(backbone_dir, shard_name), framework="pt", device="cpu"
            )
        return open_shards[shard_name].get_tensor(key)

    def maybe_get_bias(prefix: str, dev: str, dtype=torch.float16) -> Optional[torch.Tensor]:
        b_key = f"{prefix}.bias"
        if b_key in idx_map:
            return get_tensor_cpu(b_key).to(dev, dtype=dtype)
        return None

    def dequant_nf4_mos(prefix: str, dev: str, dtype=torch.float16) -> torch.Tensor:
        w_key = f"{prefix}.weight"
        q_key = f"{prefix}.weight.quant_state.bitsandbytes__nf4"
        if q_key not in idx_map:
            w_out = get_tensor_cpu(w_key).to(dev, dtype=dtype)
        else:
            w_u8 = get_tensor_cpu(w_key)
            abs_u8 = get_tensor_cpu(f"{prefix}.weight.absmax")
            nabs = get_tensor_cpu(f"{prefix}.weight.nested_absmax")
            qmap = get_tensor_cpu(f"{prefix}.weight.quant_map")
            nqmap = get_tensor_cpu(f"{prefix}.weight.nested_quant_map")
            qs_b = get_tensor_cpu(q_key)
            qs = json.loads(bytes(qs_b.numpy()).decode("utf-8"))
            shape = tuple(qs["shape"])
            bs = int(qs["blocksize"])
            nbs = int(qs["nested_blocksize"])
            noff = float(qs["nested_offset"])

            qmap_d = qmap.to(dev, dtype=torch.float32)
            nqmap_d = nqmap.to(dev, dtype=torch.float32)
            real_abs = (
                nqmap_d[abs_u8.to(dev).long()].view(-1, nbs)
                * nabs.to(dev, dtype=torch.float32).view(-1, 1)
            ).reshape(-1) + noff
            idx256 = torch.arange(256, device=dev, dtype=torch.long)
            pair_lut = torch.stack([qmap_d[idx256 >> 4], qmap_d[idx256 & 0x0F]], dim=1)

            w_flat = w_u8.reshape(-1)
            out_flat = torch.empty(shape[0] * shape[1], device=dev, dtype=dtype)
            chunk_bytes = 67108864  # 64 MB packed = 128M weights (keeps LUT temp buffer <= 512 MB!)
            for b_s in range(0, w_flat.numel(), chunk_bytes):
                b_e = min(b_s + chunk_bytes, w_flat.numel())
                c_u8 = w_flat[b_s:b_e].to(dev, non_blocking=True)
                vals = pair_lut[c_u8.to(torch.int32)].view(-1, bs)
                vals.mul_(real_abs[(b_s * 2) // bs : (b_e * 2) // bs, None])
                out_flat[b_s * 2 : b_e * 2] = vals.view(-1).to(dtype)
                del c_u8, vals
            del qmap_d, nqmap_d, real_abs, idx256, pair_lut
            w_out = out_flat.view(shape)

        # In-place fusion of proprietary MoS Residual Scar Experts (dW = scar_scale * B @ A)
        if scar_tensors is not None:
            a_key = f"{prefix}.scar_A"
            b_key = f"{prefix}.scar_B"
            if a_key in scar_tensors and b_key in scar_tensors:
                sA = scar_tensors[a_key].to(dev, dtype=torch.float32)
                sB = scar_tensors[b_key].to(dev, dtype=torch.float32)
                w_out.add_((torch.matmul(sB, sA) * scar_scale).to(dtype))
                del sA, sB
        return w_out

    clean_ids = [list(seq) if len(seq) > 0 else [pad_id] for seq in ids_list]
    n_samples = len(clean_ids)
    order = np.argsort([len(x) for x in clean_ids], kind="stable")

    micro_specs = []
    for i in range(0, n_samples, micro_bsz):
        b_idx = order[i : i + micro_bsz]
        max_s = max(len(clean_ids[j]) for j in b_idx)
        inp_ids = torch.full((len(b_idx), max_s), pad_id, dtype=torch.long)
        att_msk = torch.zeros((len(b_idx), max_s), dtype=torch.bool)
        for r, j in enumerate(b_idx):
            seq = clean_ids[j]
            L = len(seq)
            if padding_side == "left":
                inp_ids[r, max_s - L :] = torch.tensor(seq, dtype=torch.long)
                att_msk[r, max_s - L :] = True
            else:
                inp_ids[r, :L] = torch.tensor(seq, dtype=torch.long)
                att_msk[r, :L] = True
        micro_specs.append((b_idx, inp_ids, att_msk))

    max_tokens_macro = max(int(max_tokens_per_dev) * max(len(devices), 1), 1)
    macro_chunks = []
    cur_chunk, cur_toks = [], 0
    for spec in micro_specs:
        t_cnt = int(spec[1].numel())
        if cur_chunk and cur_toks + t_cnt > max_tokens_macro:
            macro_chunks.append(cur_chunk)
            cur_chunk, cur_toks = [], 0
        cur_chunk.append(spec)
        cur_toks += t_cnt
    if cur_chunk:
        macro_chunks.append(cur_chunk)

    n_slots = len(target_layers_1b)
    Z_out = np.zeros((n_samples, n_slots * d_model), dtype=np.float32)

    def rot_half(z: torch.Tensor) -> torch.Tensor:
        z1, z2 = z[..., : head_dim // 2], z[..., head_dim // 2 :]
        return torch.cat([-z2, z1], dim=-1)

    def build_rope_and_bias(att_msk_d: torch.Tensor, dev: str) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        bsz, S = att_msk_d.shape
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32, device=dev) / head_dim))
        t = torch.arange(S, dtype=torch.float32, device=dev)
        freqs = torch.outer(t, inv_freq)
        emb_pos = torch.cat([freqs, freqs], dim=-1)
        cos = emb_pos.cos().to(torch.float16)[None, None, :, :]
        sin = emb_pos.sin().to(torch.float16)[None, None, :, :]

        causal = torch.tril(torch.ones((S, S), dtype=torch.bool, device=dev))
        allow = causal[None, None, :, :] & att_msk_d[:, None, None, :]
        all_masked = ~allow.any(dim=-1, keepdim=True)
        allow = allow | (all_masked & causal[None, None, :, :])
        attn_bias = torch.zeros((bsz, 1, S, S), dtype=torch.float16, device=dev).masked_fill(
            ~allow, torch.finfo(torch.float16).min
        )
        return cos, sin, attn_bias

    def _step_attn(lp: str, active_devs: List[str], dev_states: Dict[str, list]) -> None:
        d0 = active_devs[0]
        in_ln_0 = get_tensor_cpu(f"{lp}.input_layernorm.weight").to(d0, dtype=torch.float16)
        q_0 = dequant_nf4_mos(f"{lp}.self_attn.q_proj", d0)
        k_0 = dequant_nf4_mos(f"{lp}.self_attn.k_proj", d0)
        v_0 = dequant_nf4_mos(f"{lp}.self_attn.v_proj", d0)
        o_0 = dequant_nf4_mos(f"{lp}.self_attn.o_proj", d0)
        qb_0 = maybe_get_bias(f"{lp}.self_attn.q_proj", d0) if has_qkv_bias else None
        kb_0 = maybe_get_bias(f"{lp}.self_attn.k_proj", d0) if has_qkv_bias else None
        vb_0 = maybe_get_bias(f"{lp}.self_attn.v_proj", d0) if has_qkv_bias else None
        qn_0 = get_tensor_cpu(f"{lp}.self_attn.q_norm.weight").to(d0, dtype=torch.float16) if f"{lp}.self_attn.q_norm.weight" in idx_map else None
        kn_0 = get_tensor_cpu(f"{lp}.self_attn.k_norm.weight").to(d0, dtype=torch.float16) if f"{lp}.self_attn.k_norm.weight" in idx_map else None

        attn_w = {d0: (in_ln_0, q_0, k_0, v_0, o_0, qb_0, kb_0, vb_0, qn_0, kn_0)}
        for d in active_devs[1:]:
            attn_w[d] = tuple(
                t.to(d, non_blocking=True) if t is not None else None for t in attn_w[d0]
            )
        del in_ln_0, q_0, k_0, v_0, o_0, qb_0, kb_0, vb_0, qn_0, kn_0

        max_b = max(len(dev_states[d]) for d in active_devs)
        for bi in range(max_b):
            for d in active_devs:
                if bi < len(dev_states[d]):
                    _, x, att_msk_d = dev_states[d][bi]
                    in_ln, qw, kw, vw, ow, qb, kb, vb, qn, kn = attn_w[d]
                    bsz, S, _ = x.shape
                    cos, sin, attn_bias = build_rope_and_bias(att_msk_d, d)
                    xf = x.float()
                    normed = (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * in_ln
                    del xf
                    q = F.linear(normed, qw, qb).view(bsz, S, n_heads, head_dim)
                    k = F.linear(normed, kw, kb).view(bsz, S, n_kv, head_dim)
                    v = F.linear(normed, vw, vb).view(bsz, S, n_kv, head_dim).transpose(1, 2)
                    del normed
                    if qn is not None:
                        qf = q.float()
                        q = (qf * torch.rsqrt(qf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * qn
                    if kn is not None:
                        kf = k.float()
                        k = (kf * torch.rsqrt(kf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * kn
                    q = q.transpose(1, 2)
                    k = k.transpose(1, 2)
                    q = (q * cos) + (rot_half(q) * sin)
                    k = (k * cos) + (rot_half(k) * sin)
                    k_rep = k.repeat_interleave(gqa_rep, dim=1) if gqa_rep > 1 else k
                    v_rep = v.repeat_interleave(gqa_rep, dim=1) if gqa_rep > 1 else v
                    ao = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=attn_bias)
                    dev_states[d][bi][1] = x + F.linear(
                        ao.transpose(1, 2).contiguous().view(bsz, S, n_heads * head_dim), ow
                    )
                    del cos, sin, attn_bias, q, k, v, k_rep, v_rep, ao
        del attn_w

    def _step_mlp(lp: str, active_devs: List[str], dev_states: Dict[str, list]) -> None:
        d0 = active_devs[0]
        post_ln_0 = get_tensor_cpu(f"{lp}.post_attention_layernorm.weight").to(d0, dtype=torch.float16)
        gate_0 = dequant_nf4_mos(f"{lp}.mlp.gate_proj", d0)
        up_0 = dequant_nf4_mos(f"{lp}.mlp.up_proj", d0)
        down_0 = dequant_nf4_mos(f"{lp}.mlp.down_proj", d0)

        mlp_w = {d0: (post_ln_0, gate_0, up_0, down_0)}
        for d in active_devs[1:]:
            mlp_w[d] = tuple(t.to(d, non_blocking=True) for t in mlp_w[d0])
        del post_ln_0, gate_0, up_0, down_0

        max_b = max(len(dev_states[d]) for d in active_devs)
        for bi in range(max_b):
            for d in active_devs:
                if bi < len(dev_states[d]):
                    x = dev_states[d][bi][1]
                    pln, gw, uw, dw = mlp_w[d]
                    xf = x.float()
                    normed = (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * pln
                    del xf
                    act = F.silu(F.linear(normed, gw))
                    act.mul_(F.linear(normed, uw))
                    del normed
                    dev_states[d][bi][1] = x + F.linear(act, dw)
                    del act
        del mlp_w

    def _pool_slot(slot_i: int, apply_final_norm: bool, final_ln_cpu: torch.Tensor, active_devs: List[str], dev_states: Dict[str, list]) -> None:
        col_s = slot_i * d_model
        col_e = col_s + d_model
        for d in active_devs:
            fln = final_ln_cpu.to(d, dtype=torch.float16) if apply_final_norm else None
            for b_idx, x, att_msk_d in dev_states[d]:
                xf = x.float()
                if fln is not None:
                    x_use = ((xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * fln).float()
                else:
                    x_use = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)
                m = att_msk_d.unsqueeze(-1).float()
                pooled = (x_use * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
                rms = torch.rsqrt(pooled.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
                Z_out[b_idx, col_s:col_e] = (pooled * rms).cpu().numpy()
                del xf, x_use, m, pooled, rms
            del fln

    t_mos0 = time.time()
    with torch.no_grad():
        for mc_idx, m_specs in enumerate(macro_chunks):
            open_shards.clear()
            emb_w = get_tensor_cpu(f"{base_pfx}embed_tokens.weight")
            dev_states = {d: [] for d in devices}
            for bi, (b_idx, inp_ids, att_msk) in enumerate(m_specs):
                d = devices[bi % len(devices)]
                x = F.embedding(inp_ids, emb_w).to(d, dtype=torch.float16)
                dev_states[d].append([b_idx, x, att_msk.to(d)])
            del emb_w
            active_devs = [d for d in devices if len(dev_states[d]) > 0]
            final_ln_cpu = get_tensor_cpu(norm_key)

            for l in range(max_target_layer):
                lp = f"{base_pfx}layers.{l}"
                needed_from_l = {
                    idx_map[k]
                    for k in idx_map
                    if k == norm_key
                    or (k.startswith(layers_pfx) and int(k[len(layers_pfx) :].split(".")[0]) >= l)
                }
                for s_name in list(open_shards.keys()):
                    if s_name not in needed_from_l:
                        del open_shards[s_name]

                _step_attn(lp, active_devs, dev_states)
                _step_mlp(lp, active_devs, dev_states)

                if l in capture_map:
                    _pool_slot(
                        capture_map[l],
                        apply_final_norm=(l == n_layers - 1),
                        final_ln_cpu=final_ln_cpu,
                        active_devs=active_devs,
                        dev_states=dev_states,
                    )

                if (l + 1) % log_every == 0 or (l + 1) == max_target_layer:
                    gc.collect()
                    vram_gb = (
                        torch.cuda.memory_allocated(0) / (1024 ** 3)
                        if torch.cuda.is_available()
                        else 0.0
                    )
                    print(
                        f"  [MoS Stream - {tag}] Macro {mc_idx+1}/{len(macro_chunks)} | "
                        f"Layer {l+1}/{max_target_layer} | Elapsed {time.time()-t_mos0:.1f}s | "
                        f"GPU0 Post-Layer Alloc {vram_gb:.2f} GiB",
                        flush=True,
                    )

            del dev_states, final_ln_cpu
            open_shards.clear()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return Z_out
train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)


In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

SMALL_WORDS_C2 = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]

def shape_text(t):
    """Every run of letters becomes X or x, so only the rhythm is left."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)

def small_words_text(t):
    """Keep the small words and the punctuation, and replace every other word by CONTENT."""
    return " ".join(w.lower() if w.lower() in SMALL_WORDS_C2 or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))

SOURCE_CODE_SHA256 = "5864b360ef4b470726b6ae2018489e285651dfb2a305d24b1c6884666d81f800"
import hashlib, importlib.metadata, joblib
from pathlib import Path
EXPECTED_CACHE_SHA256 = "f294ce929714e852a5a3f1d34ff7a5c5f0a0df3d73bd4a24ae327cbde4d96352"
def _sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
    return h.hexdigest()
def _one(pattern):
    hits=sorted(set(glob.glob(f"{INPUT_DIR}/**/{pattern}",recursive=True)))
    if len(hits)!=1: raise RuntimeError(("Ambiguous or absent input",pattern,hits))
    return Path(hits[0])
mp = _one("ntl620_svc_prefit_v1/manifest.json")
manifest=json.loads(mp.read_text())
assert EXPECTED_CACHE_SHA256 != "SET_FROM_CPU_MANIFEST_BEFORE_GPU", "CPU cache checksum not pinned"
assert manifest["schema"]=="ntl620-svc-prefit-v1" and manifest["source_version"]==361311282
assert manifest["source_code_sha256"] == SOURCE_CODE_SHA256
assert manifest["labels"]==LABELS and manifest["views"]==[list(v) for v in VIEWS]
assert manifest["classes"]==list(range(K)) and not manifest["fit_used_test_rows"]
assert manifest["sample_cache_reload_exact"] is True
assert _sha(_one("train.csv"))==manifest["train_sha256"]
assert _sha(_one("pool_v4.parquet"))==manifest["pool_sha256"]
for package, expected in manifest["versions"].items():
    assert importlib.metadata.version(package)==expected, (package,expected,importlib.metadata.version(package))
cp=mp.parent/"svc_prefit.joblib"
assert manifest["cache_sha256"]==EXPECTED_CACHE_SHA256 and _sha(cp)==EXPECTED_CACHE_SHA256
cache=joblib.load(cp)  # checksum-pinned statistical anchor
assert cache["labels"]==LABELS and cache["svc"].classes_.tolist()==list(range(K))
B=[]
for vec,(view,lo,hi,max_features,weight) in zip(cache["vectorizers"],VIEWS):
    prep=shape_text if view=="shape" else small_words_text if view=="small" else (lambda s:s)
    B.append(vec.transform([prep(t) for t in test.text])*weight)
assert len(cache["vectorizers"])==len(VIEWS)==4
vote("svc_open_pool", softmax(cache["svc"].decision_function(hstack(B,format="csr")) / 0.85), W_SVC)
del B, cache
gc.collect()


In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

if len(test) >= 2:
    Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
    S_mat = (Xt @ Xt.T).toarray().astype(np.float32)
    del Xt
    D = np.clip(1.0 - S_mat, 0.0, None)
    np.fill_diagonal(D, 0.0)
    Z_lk = linkage(squareform(D, checks=False), method="average")
    g_raw = fcluster(Z_lk, TWIN["thresh"], criterion="distance")
    next_id = int(g_raw.max()) + 1
    groups = g_raw.copy()
    for cid in np.unique(g_raw):
        idx = np.where(g_raw == cid)[0]
        if len(idx) > TWIN["max_size"]:
            subD = D[np.ix_(idx, idx)]
            subZ = linkage(squareform(subD, checks=False), method="average")
            for th in np.linspace(TWIN["thresh"] - 0.005, 0.05, 165):
                sg = fcluster(subZ, th, criterion="distance")
                if np.bincount(sg).max() <= TWIN["max_size"]:
                    for sc in np.unique(sg):
                        groups[idx[sg == sc]] = next_id; next_id += 1
                    break
            else:
                sg = fcluster(subZ, int(np.ceil(len(idx) / TWIN["max_size"])), criterion="maxclust")
                for sc in np.unique(sg):
                    groups[idx[sg == sc]] = next_id; next_id += 1
    del D, Z_lk
else:
    groups = np.ones(len(test), dtype=int)

size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
print("topic clusters:", (size == 1).sum(), "passages alone,", (twin_rows).sum(), "in clusters of 2 to 18", flush=True)


In [ ]:
from transformers import AutoModel, AutoTokenizer

ck = os.path.dirname(find("deberta_v3_large/meta.json"))
if ck.startswith("/kaggle/input"):
    shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
    ck = "/tmp/deberta_v3_large"
meta = json.load(open(os.path.join(ck, "meta.json")))
assert meta["labels"] == LABELS, "label order differs from checkpoint head"
etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
head = load_file(os.path.join(ck, "head.safetensors"))
W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
ids = etok(test.text.tolist(), truncation=True, max_length=meta["max_len"])["input_ids"]
order = np.argsort([len(x) for x in ids], kind="stable")
reps = [None] * len(ids)
with torch.no_grad():
    for i in range(0, len(ids), 32):
        b_idx = order[i:i + 32]
        b_tok = etok.pad({"input_ids": [ids[j] for j in b_idx]}, padding=True, return_tensors="pt").to("cuda:0")
        h = enc(**b_tok).last_hidden_state
        m = b_tok["attention_mask"][..., None]
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        for j, r in zip(b_idx, pooled.cpu().numpy()):
            reps[j] = r
        del b_tok, h, m, pooled
vote("deberta", softmax(np.stack(reps) @ W.T + b), W_DEBERTA)
del enc, head, reps
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
head_mistral_file = find("head_tricompass_mistral_large_123b_l44_scache.safetensors")
head_mistral = load_file(head_mistral_file)
mistral_backbone_dir = os.path.dirname(find("model-00001-of-00013.safetensors"))
tok_mistral = AutoTokenizer.from_pretrained(mistral_backbone_dir, use_fast=True)
pad_id_m = tok_mistral.pad_token_id if tok_mistral.pad_token_id is not None else tok_mistral.eos_token_id
raw_ids_m = tok_mistral(test.text.tolist(), truncation=True, max_length=512, padding=False)["input_ids"]
mos_devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else ["cpu"]

Z_mistral = extract_mos_nf4_streaming(
    mistral_backbone_dir, raw_ids_m, pad_id_m, mos_devices,
    padding_side=tok_mistral.padding_side, tag="Titan3-Mistral123B",
    micro_bsz=8, max_tokens_per_dev=60000,
)
assert np.isfinite(Z_mistral).all(), "Non-finite features in Z_mistral"

style_raw = extract_tricompass_style_features(test.text.tolist())
style_normed = (style_raw - head_mistral["style_scaler.mean"].float().numpy()) / head_mistral["style_scaler.scale"].float().numpy()
X_tri_mistral = np.hstack([Z_mistral, style_normed])
X_sc_mistral = TestTimeSCache(slot_dim=32, resonance_scale=0.15).apply_scache_resonance(X_tri_mistral, groups, twin_rows)

X_plain = (X_sc_mistral - head_mistral["scaler_plain.mean"].float().numpy()) / head_mistral["scaler_plain.scale"].float().numpy()
logits_plain = (X_plain @ head_mistral["plain.weight"].float().numpy().T + head_mistral["plain.bias"].float().numpy()) / TEMP_TITAN3
vote("titan3_mistral_123b_plain", softmax(logits_plain), W_TITAN3_PLAIN)

if twin_rows.any():
    X_c_raw = X_sc_mistral.copy()
    for g in np.unique(groups[twin_rows]):
        m = (groups == g)
        if m.sum() > 1:
            X_c_raw[m] -= X_c_raw[m].mean(axis=0, keepdims=True)
    X_c = (X_c_raw - head_mistral["scaler_centered.mean"].float().numpy()) / head_mistral["scaler_centered.scale"].float().numpy()
    logits_c = (X_c[twin_rows] @ head_mistral["centered.weight"].float().numpy().T + head_mistral["centered.bias"].float().numpy()) / TEMP_TITAN3
    vote("titan3_mistral_123b_twin", softmax(logits_c), W_TITAN3_TWIN, rows=twin_rows)

del Z_mistral, X_tri_mistral, X_sc_mistral, X_plain, head_mistral
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def sinkhorn_damped(P, target_mix=0.50, iters=300):
    P_s = np.clip(P, 1e-12, 1.0)
    P_s = P_s / P_s.sum(1, keepdims=True)
    emp = P_s.sum(0)
    unif = np.full(K, len(P_s) / float(K))
    target = np.maximum((1.0 - target_mix) * emp + target_mix * unif, 1e-6)
    c = np.ones(K, dtype=np.float64)
    for _ in range(iters):
        R = P_s * c
        R /= R.sum(1, keepdims=True)
        c *= target / np.maximum(R.sum(0), 1e-12)
    R = P_s * c
    return R / R.sum(1, keepdims=True)

assert np.isfinite(LOGIT).all(), "Nonfinite ensemble logits"
P = softmax(LOGIT)
if len(test) >= 50:
    P = sinkhorn_damped(P, target_mix=0.50, iters=300)
assert np.isfinite(P).all(), "Nonfinite ensemble probability"
sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
assert len(sub) == len(test) and sub["model"].isin(LABELS).all()
sub.to_parquet("submission.parquet", index=False)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")
